generating prior samples for a particular model.  using as a way to check progress of the model

In [ ]:
from sigpy.plot import ImagePlot as iplt
%matplotlib widget

import torch
from torch_utils import distributed as dist
import dnnlib
import pickle
import matplotlib.pyplot as mplt
import numpy as np

from sampling_funcs import StackedRandomGenerator,ODE_motion_sampler

In [ ]:
num_samples = 1  # number of samples that we want to generate
num_batch   = 32 # bath size
M = 192
N = 192 # image dimensions

#- not sure what this is for
class_labels = None

#- sampling parameters
num_steps = 300
sigma_max = 5.0
sigma_min = 0.002

rho = 7 # for time-step discretization

In [ ]:
device = 'cuda:2'

# load network
# net_save = '/csiNAS2/slow/yarefeen/edm-outputs-test/00006-aspect_preprocessed-uncond-ddpmpp-edm-gpus1-batch25-fp32-aspect-preprocessed-test-1/network-snapshot-001050.pkl'
net_save = '/csiNAS2/slow/brett/edm_outputs/00036-aspect_preprocessed-uncond-ddpmpp-edm-gpus4-batch40-fp32-aspect/network-snapshot-010000.pkl' # change to YOUR path
if dist.get_rank() != 0:
        torch.distributed.barrier()
        
dist.print0(f'Loading network from "{net_save}"...')
with dnnlib.util.open_url(net_save, verbose=(dist.get_rank() == 0)) as f:
    net = pickle.load(f)['ema'].to(device)

##### performing the sampling

In [ ]:
#-setting time-step discretization
step_indices = torch.arange(num_steps, dtype=torch.float64, device=device)
t_steps = (sigma_max ** (1 / rho) + step_indices / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))) ** rho
t_steps = torch.cat([net.round_sigma(t_steps), torch.zeros_like(t_steps[:1])]) # t_N = 0

all_x = np.zeros((num_batch,M,N,num_steps,num_samples),dtype = complex)

# - should probably do this in batches later
for num_sample in range(num_samples):
    print('SAMPLE: %d / %d' % (num_sample+1, num_samples))
    rnd = StackedRandomGenerator(device, [1]) # use the current sample number as the seeda
    
    #-generating the gaussian sample which will be 'denoised' to our desired sample
#     latents = rnd.randn([1, 2, M, N], device=device)
    latents = torch.randn((num_batch,2,M,N),device=device)
    x = latents.to(torch.float64) * t_steps[0] 
    # scale initial noise image to have same variance as starting sigma
    
    for i, (t_cur, t_next) in enumerate(zip(t_steps[:-1], t_steps[1:])): # 0, ..., N-1 # for tqdm
        print('   step: %d/%d' % (i+1,num_steps))
        #-getting scoare with denoising
        denoised = net(x, t_cur, class_labels).to(torch.float64)

        score = (x - denoised) / t_cur
        
        #-performing the update
        x = x + (t_next - t_cur) * score
        all_x[:,:,:,i,num_sample] = torch.view_as_complex(x.permute(0,-2,-1,1).contiguous()).squeeze().cpu()


In [ ]:
all_x_rsh = np.transpose(all_x,axes=(0,4,1,2,3))
all_x_rsh = np.reshape(all_x_rsh,(num_samples*num_batch,M,N,num_steps))

In [ ]:
print(all_x_rsh.shape)

In [ ]:
print(np.max(np.abs(all_x_rsh[:,:,:,-1].flatten())))

In [ ]:
iplt(np.abs(all_x_rsh[:,:,:,-1]),vmax=1/3)

In [ ]:
iplt(np.abs(all_x_rsh),x=1,y=2,vmax=.1)